In [78]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

In [80]:
# fetch dataset 
multivariate_gait_data = fetch_ucirepo(id=760)
df = multivariate_gait_data.data.features

### LSTM

In [120]:
# randomly select training/testing subjects (80/20 split)
# split data by subjects to avoid data leakage
train_subjects = random.sample(range(1, 11), 8)
test_subjects = [i for i in range(1, 11) if i not in train_subjects]

train = df[df["subject"].isin(train_subjects)]
test = df[df["subject"].isin(test_subjects)]

In [124]:
def construct_sequences(df):
    """ 
    Construct sequences of leg/joint angles over time 
    Each sequence has shape 101 x 6 (101 points in time x 6 angles)
    """
    grouped = df.groupby(["subject", "condition", "replication"])

    labels = []
    sequences = []

    for id, group in grouped:
        # condition at index 1 in the groupby
        labels.append(id[1])
        # get measurements for each leg and joint combination over time
        pivot = group.pivot_table(
            index="time", 
            columns=["leg", "joint"], 
            values="angle"
        )
        sequences.append(pivot.values)

    sequences = np.array(sequences)
    labels = np.array(labels)

    return sequences, labels

In [153]:
# format the data for training and testing
X_train, y_train = construct_sequences(train)
X_test, y_test = construct_sequences(test)

# flatten the data to apply standardization
n_instances, n_timepts, n_features = X_train.shape
X_train_flat = X_train.reshape(-1, n_features)
X_test_flat = X_test.reshape(-1, n_features)

# scale the data and reshape it into a 3D tensor
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_flat).reshape(n_instances, n_timepts, n_features)
X_test = scaler.transform(X_test_flat).reshape(-1, n_timepts, n_features)

# 0-indexed labels
y_train = y_train - 1
y_test = y_test - 1